In [1]:
import poolparty as pp
import tempfile
import os

# Initialize poolparty
pp.init()

# Create a simple library without configuration (shows all columns)
print("=== Without configuration (all columns visible) ===")
pool1 = pp.from_seqs(['ACGT', 'TGCA', 'GGGG'], seq_names=['seq1', 'seq2', 'seq3']).mutagenize(num_mutations=1)
df1 = pool1.generate_library(num_seqs=3, report_design_cards=True)
print(df1.columns.tolist())
print()

# Create a TOML config that hides some columns
config_content = """
[columns]
name = true
seq = true
pool_seqs = false     # Hide pool-specific seq columns
pool_states = false   # Hide pool state columns
op_states = false     # Hide operation state columns

[design_cards.from_seqs]
seq_name = true       # Show seq_name
seq_index = false     # Hide seq_index

[design_cards.mutagenize]
positions = true      # Show positions
wt_chars = false      # Hide wt_chars
mut_chars = false     # Hide mut_chars
"""

# Write config to temp file and load it
with tempfile.NamedTemporaryFile(mode='w', suffix='.toml', delete=False) as f:
    f.write(config_content)
    temp_path = f.name

pp.load_config(temp_path)

print("=== With configuration (filtered columns) ===")
pool2 = pp.from_seqs(['ACGT', 'TGCA', 'GGGG'], seq_names=['seq1', 'seq2', 'seq3']).mutagenize(num_mutations=1)
df2 = pool2.generate_library(num_seqs=3, report_design_cards=True)
print(df2.columns.tolist())
print()
print(df2)

# Clean up
os.unlink(temp_path)

print("\n=== Configuration successfully controls column visibility! ===")

=== Without configuration (all columns visible) ===
['name', 'seq', 'pool[1].seq', 'pool[0].seq', 'op[1]:mutagenize.key.positions', 'op[1]:mutagenize.key.wt_chars', 'op[1]:mutagenize.key.mut_chars', 'op[0]:from_seqs.key.seq_name', 'op[0]:from_seqs.key.seq_index']

=== With configuration (filtered columns) ===
['name', 'seq', 'op[3]:mutagenize.key.positions', 'op[2]:from_seqs.key.seq_name']

   name   seq op[3]:mutagenize.key.positions op[2]:from_seqs.key.seq_name
0  seq3  GGCG                           (2,)                         seq3
1  seq1  AAGT                           (1,)                         seq1
2  seq1  CCGT                           (0,)                         seq1

=== Configuration successfully controls column visibility! ===
